In [1]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
from pathlib import Path
import sqlite3
import matplotlib.pyplot as plt

db_path = Path("../db.sqlite3")
if not db_path.exists():
    db_path = Path("db.sqlite3")

connection = sqlite3.connect(db_path)

query = """
SELECT
    order_item.id AS "Order Item ID",
    order_item.order_id AS "Order ID",
    orders.datetime AS "Order Datetime",
    order_item.product_id AS "Product ID",
    product.name AS "Product Name",
    product.price AS "Product Price",
    order_item.quantity AS "Quantity"
FROM restaurant_orderitem AS order_item
JOIN restaurant_order AS orders
    ON order_item.order_id = orders.id
JOIN restaurant_product AS product
    ON order_item.product_id = product.id
"""

all_data = pd.read_sql_query(query, connection)
connection.close()

all_data.to_csv("restaurant_data.csv", index=False)

df = pd.read_csv("restaurant_data.csv")
df["Order Datetime"] = pd.to_datetime(df["Order Datetime"])
df["Product Price"] = pd.to_numeric(df["Product Price"])

print(df.shape)
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_products_by_quantity = (
    df.groupby("Product Name")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 8))
plt.pie(
    top_products_by_quantity,
    labels=top_products_by_quantity.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Top 10 products by quantity")
plt.axis("equal")
plt.show()

top_products_by_quantity

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["Item Price"] = df["Product Price"] * df["Quantity"]

top_products_by_income = (
    df.groupby("Product Name")["Item Price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 8))
plt.pie(
    top_products_by_income,
    labels=top_products_by_income.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Top 10 products by income")
plt.axis("equal")
plt.show()

top_products_by_income

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["Order Hour"] = df["Order Datetime"].dt.hour

income_by_hour = (
    df.groupby("Order Hour")["Item Price"]
    .sum()
    .reindex(range(24), fill_value=0)
)

plt.figure(figsize=(12, 6))
plt.bar(income_by_hour.index, income_by_hour.values)
plt.xticks(range(24))
plt.xlabel("Order Hour")
plt.ylabel("Total Income")
plt.title("Restaurant income by order hour")
plt.show()

income_by_hour

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
days_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

df["Order Day Of The Week"] = df["Order Datetime"].dt.day_name()

income_by_day = (
    df.groupby("Order Day Of The Week")["Item Price"]
    .sum()
    .reindex(days_order, fill_value=0)
)

plt.figure(figsize=(10, 6))
plt.bar(income_by_day.index, income_by_day.values)
plt.xlabel("Order Day Of The Week")
plt.ylabel("Total Income")
plt.title("Restaurant income by day of the week")
plt.xticks(rotation=45)
plt.show()

income_by_day